# HELICoiD: SegFormer With and Without PCA

This notebook tests the PCA step of the current study on the **HELICoiD** brain-tumour hyperspectral dataset, using the same SegFormer model. Run it on **Google Colab with a GPU** (Runtime → Run all). See `HOW_TO_RUN.md`, Part 5.

HELICoiD differs from the melanoma data in one important way: it comes with **expert labels** for four classes (normal tissue, tumour, blood vessel, background). Only some pixels are labelled. So the notebook runs two parts:

| Part | Question | How |
|---|---|---|
| **1** | Does the current automatic labelling (SLIC + KMeans) find tumour, with and without PCA? | Pseudo labels are compared with the expert tumour labels. No training. |
| **2** | Does SegFormer segment better with PCA input or with all bands? | SegFormer is trained on the expert labels, once with 3 PCA channels and once with all bands, using patient-wise cross-validation. |

The current pipeline's labelling assumes one lesion near the image centre, which fits skin images but not brain surgery images. That is why Part 2 trains on the expert labels instead.

## Colab setup

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers<5', 'accelerate', 'albumentations'], check=True)
else:
    print('Not on Colab. This notebook is written for Google Colab.')

In [ ]:
import glob, json, re, time, shutil, zipfile, posixpath, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold
from skimage.segmentation import slic
from skimage.morphology import opening, closing, disk
from skimage.measure import label, regionprops

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation, TrainingArguments, Trainer, set_seed

warnings.filterwarnings("ignore")

## 0. Settings

In [ ]:
# ---- Where the data is ----
DRIVE_DATA_DIR = '/content/drive/MyDrive/dataset_helicoid'   # shortcut to the shared folder
ZIP_NAMES = ['FirstCampaign.zip', 'SecondCampaign.zip', 'ThirdCampaign.zip']
DRIVE_FOLDER_NAME = 'dataset_helicoid'   # name of the shared folder (or your shortcut to it)
DRIVE_FOLDER_ID = ''                    # optional: the ID from the folder's link, if the search fails

# ---- Working folders on Colab's own disk ----
LOCAL_ZIP_DIR = '/content/helicoid_zips'
WORK_DIR = '/content/helicoid_data'        # processed images
OUT_DIR = '/content/helicoid_results'      # tables and charts
SAVE_DIR = '/content/drive/MyDrive/HELICoiD_PCA_outputs'   # results are copied here at the end

# ---- Data preparation ----
N_BANDS = 128            # spectral bands kept after averaging neighbouring bands
CLASS_NAMES = {1: 'Normal tissue', 2: 'Tumour', 3: 'Blood vessel', 4: 'Background'}   # label value -> name; 0 = unlabelled
TUMOUR_CLASS = 2

# ---- Training settings, same as the current study ----
PRETRAINED = 'nvidia/segformer-b0-finetuned-ade-512-512'
IMG_SIZE = 256
EPOCHS = 20
BATCH_SIZE = 2
LEARNING_RATE = 5e-5
SEED = 42
N_FOLDS = 5              # patient-wise cross-validation folds

# ---- Pseudo-label settings, same as the melanoma comparison ----
SLIC_COMPACTNESS_NO_PCA = 1

for d in [LOCAL_ZIP_DIR, WORK_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cpu':
    print('WARNING: no GPU. Choose Runtime > Change runtime type > T4 GPU, then run again.')

## 1. Copy the zip files to Colab's disk

Reading large files straight from Google Drive can drop the connection, so each zip is copied with retries. Re-running skips zips that are already copied.

In [ ]:
def with_retry(action, what, tries=6):
    for attempt in range(1, tries + 1):
        try:
            return action()
        except OSError as e:
            if attempt == tries:
                raise RuntimeError(
                    f"Google Drive kept failing while reading {what}. Run this cell again. If it fails every time, "
                    "Drive may be limiting downloads of this shared file: in Google Drive, right-click the zip, "
                    "choose 'Make a copy', and put the copies in the dataset_helicoid shortcut's place.") from e
            print(f"  Drive problem on {what}, reconnecting (attempt {attempt} of {tries - 1})")
            time.sleep(10)
            if IN_COLAB:
                drive.mount('/content/drive', force_remount=True)


def copy_one(src, dst):
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return False
    shutil.copyfile(src, dst + '.part')
    os.replace(dst + '.part', dst)
    return True


def find_zip_folder():
    # Fast path: the zips are visible through the mounted Drive folder.
    candidates = [DRIVE_DATA_DIR] + glob.glob('/content/drive/MyDrive/' + DRIVE_FOLDER_NAME + '*') \
                 + glob.glob('/content/drive/.shortcut-targets-by-id/*/' + DRIVE_FOLDER_NAME + '*')
    for c in candidates:
        try:
            if os.path.exists(os.path.join(c, ZIP_NAMES[0])):
                return c
        except OSError:
            pass
    for c in candidates[1:]:
        try:
            print(f"  Checked {c}: link={os.path.islink(c)}, contents={sorted(os.listdir(c))[:10]}")
        except OSError as e:
            print(f"  Checked {c}: cannot open ({e})")
    return None


def download_with_drive_api():
    # Download the zips straight from Google Drive with your account. Works for shared folders and shortcuts.
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    from googleapiclient.errors import HttpError
    auth.authenticate_user()
    service = build('drive', 'v3')

    def list_files(query):
        found, token = [], None
        while True:
            r = service.files().list(q=query, pageSize=100, pageToken=token, supportsAllDrives=True,
                                     includeItemsFromAllDrives=True,
                                     fields='nextPageToken, files(id, name, size, mimeType, shortcutDetails(targetId))').execute()
            found += r.get('files', [])
            token = r.get('nextPageToken')
            if not token:
                return found

    folder_ids = [DRIVE_FOLDER_ID] if DRIVE_FOLDER_ID else []
    for f in list_files(f"name = '{DRIVE_FOLDER_NAME}' and trashed = false"):
        if f['mimeType'] == 'application/vnd.google-apps.folder':
            folder_ids.append(f['id'])
        elif f['mimeType'] == 'application/vnd.google-apps.shortcut':
            folder_ids.append(f['shortcutDetails']['targetId'])
    zips = {}
    for fid in dict.fromkeys(folder_ids):
        for f in list_files(f"'{fid}' in parents and trashed = false"):
            if f['name'] in ZIP_NAMES:
                zips.setdefault(f['name'], f)
    for z in ZIP_NAMES:
        if z not in zips:
            hits = list_files(f"name = '{z}' and trashed = false")
            if hits:
                zips[z] = hits[0]
    missing = [z for z in ZIP_NAMES if z not in zips]
    if missing:
        raise FileNotFoundError(f"Google Drive search could not find {missing}. Open the shared folder in your browser, "
                                "copy the long ID after 'folders/' in its address, paste it into DRIVE_FOLDER_ID in "
                                "Section 0, and run again.")

    for name in ZIP_NAMES:
        f, dst = zips[name], os.path.join(LOCAL_ZIP_DIR, name)
        size = int(f.get('size', 0))
        if os.path.exists(dst) and os.path.getsize(dst) == size:
            print(f"{name}: already downloaded")
            continue
        start = time.time()
        try:
            with open(dst + '.part', 'wb') as fh:
                downloader = MediaIoBaseDownload(fh, service.files().get_media(fileId=f['id'], supportsAllDrives=True),
                                                 chunksize=256 * 1024 * 1024)
                done, shown = False, -1
                while not done:
                    status, done = downloader.next_chunk(num_retries=8)
                    pct = int(status.progress() * 100) if status else 100
                    if pct // 20 != shown // 20:
                        print(f"  {name}: {pct}%")
                        shown = pct
        except HttpError as e:
            raise RuntimeError(f"Google Drive refused the download of {name}: {e}. If it mentions a quota, wait a few "
                               "hours or ask the owner to share the files with you directly.") from e
        os.replace(dst + '.part', dst)
        print(f"{name}: {os.path.getsize(dst) / 1e9:.2f} GB downloaded in {time.time() - start:.0f} s")


free_gb = shutil.disk_usage('/content' if IN_COLAB else '.').free / 1e9
print(f"Free disk space: {free_gb:.0f} GB")
if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)   # picks up shortcuts added after the first mount
DATA_DIR = with_retry(find_zip_folder, 'searching Google Drive')

if DATA_DIR:
    print(f"Dataset found in: {DATA_DIR}")
    for zname in ZIP_NAMES:
        src, dst = os.path.join(DATA_DIR, zname), os.path.join(LOCAL_ZIP_DIR, zname)
        start = time.time()
        copied = with_retry(lambda: copy_one(src, dst), zname)
        size = os.path.getsize(dst) / 1e9
        print(f"{zname}: {size:.2f} GB " + (f"copied in {time.time() - start:.0f} s" if copied else "already copied"))
else:
    print("The zips are not visible through the Drive folder, so they will be downloaded with the Google Drive API.")
    print("A Google sign-in window will open: choose your account and allow access.")
    download_with_drive_api()

for zname in ZIP_NAMES:
    path = os.path.join(LOCAL_ZIP_DIR, zname)
    assert os.path.exists(path) and zipfile.is_zipfile(path), f"{zname} is missing or damaged. Run this cell again."
print("All zip files are ready.")

## 2. Look inside the zip files

This lists what each zip contains, unpacks the per-image zips inside them, and finds the image folders. An image folder is one that holds a ground-truth map (`gt…​.hdr`) and a hyperspectral cube (`.hdr` plus data file). **If this cell finds 0 images, copy its whole output and send it, so the reader can be adjusted.**

In [ ]:
def capture_roles(files, sizes):
    # Decide which files in one folder are the ground truth, references and cube.
    hdrs = [f for f in files if f.lower().endswith('.hdr')]
    gt = next((f for f in hdrs if 'gt' in f.lower()), None)
    white = next((f for f in hdrs if 'white' in f.lower()), None)
    dark = next((f for f in hdrs if 'dark' in f.lower()), None)
    others = [f for f in hdrs if f not in (gt, white, dark)]
    raw = next((f for f in others if f.lower().startswith('raw')), None)
    data_size = lambda h: max([sizes.get(f, 0) for f in files if f != h and f.startswith(h[:-4])] or [0])
    cube = raw or (max(others, key=data_size) if others else None)
    return {'gt': gt, 'white': white, 'dark': dark, 'cube': cube, 'is_raw': cube is not None and cube == raw}


INNER_DIR = os.path.join(WORK_DIR, 'inner_zips')


def scan_zip(zip_path):
    # Return the files in a zip, grouped by folder, and the folders that hold one image each.
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if not i.is_dir()]
    by_dir, sizes = defaultdict(list), defaultdict(dict)
    for i in infos:
        d, b = posixpath.split(i.filename)
        by_dir[d].append(b)
        sizes[d][b] = i.file_size
    found = []
    for d in sorted(by_dir):
        roles = capture_roles(by_dir[d], sizes[d])
        if roles['gt'] and roles['cube']:
            found.append((d, by_dir[d], roles))
    return infos, by_dir, found


CAPTURES, used_names = [], set()

def add_capture(zip_path, d, files, roles, campaign, fallback):
    capture = posixpath.basename(d) or fallback
    name, k = f"{campaign}__{capture}", 2
    while name in used_names:
        name, k = f"{campaign}__{capture}_{k}", k + 1
    used_names.add(name)
    CAPTURES.append({'zip': zip_path, 'folder': d, 'files': files, 'roles': roles,
                     'campaign': campaign, 'capture': capture, 'name': name})


for zname in ZIP_NAMES:
    campaign = zname[:-4]
    outer = os.path.join(LOCAL_ZIP_DIR, zname)
    infos, by_dir, found = scan_zip(outer)
    for d, files, roles in found:
        add_capture(outer, d, files, roles, campaign, campaign)
    kinds = pd.Series([posixpath.splitext(b)[1].lower() or '(no extension)' for d in by_dir for b in by_dir[d]]).value_counts()
    print(f"\n=== {zname}: {len(infos)} files in {len(by_dir)} folders")
    print("File types:", dict(kinds.head(8)))

    # Notes from the dataset authors (README files)
    with zipfile.ZipFile(outer) as zf:
        for info in [i for i in infos if i.filename.lower().endswith(('.md', '.txt'))]:
            text = zf.read(info).decode('utf-8', errors='ignore').strip()
            print(f"\n--- {posixpath.basename(info.filename)} (first lines) ---")
            print("\n".join(text.splitlines()[:40])[:3000])
            print("--- end ---")

    # Images packed as one zip each inside the campaign zip
    inner = [i for i in infos if i.filename.lower().endswith('.zip')]
    if inner:
        print(f"Unpacking {len(inner)} inner zip files to Colab's disk...")
        os.makedirs(os.path.join(INNER_DIR, campaign), exist_ok=True)
        start, before = time.time(), len(CAPTURES)
        with zipfile.ZipFile(outer) as zf:
            for k, info in enumerate(inner):
                dst = os.path.join(INNER_DIR, campaign, posixpath.basename(info.filename))
                if not (os.path.exists(dst) and os.path.getsize(dst) == info.file_size):
                    with zf.open(info) as src, open(dst + '.part', 'wb') as out:
                        shutil.copyfileobj(src, out, 16 * 1024 * 1024)
                    os.replace(dst + '.part', dst)
                _, inner_dirs, inner_found = scan_zip(dst)
                for d, files, roles in inner_found:
                    add_capture(dst, d, files, roles, campaign, posixpath.basename(info.filename)[:-4])
                if k == 0:
                    for d in sorted(inner_dirs)[:3]:
                        print(f"  inside {posixpath.basename(info.filename)}, folder '{d}': {sorted(inner_dirs[d])[:12]}")
        print(f"  {len(CAPTURES) - before} images found in the inner zips ({time.time() - start:.0f} s)")
    elif not found:
        for d in sorted(by_dir)[:3]:
            print(f"  example folder '{d}': {sorted(by_dir[d])[:10]}")

print(f"\nTotal images found: {len(CAPTURES)}")
if CAPTURES:
    c = CAPTURES[0]
    roles = c['roles']
    print(f"First image: {c['name']}")
    print(f"  cube: {roles['cube']} (raw, will be calibrated: {roles['is_raw'] and bool(roles['white'] and roles['dark'])})")
    print(f"  ground truth: {roles['gt']}, white: {roles['white']}, dark: {roles['dark']}")
assert CAPTURES, "No images found. Copy this cell's output and send it so the reader can be adjusted."

## 3. Prepare each image

For each image: read the cube, calibrate it with the white and dark references (when present), average neighbouring bands down to `N_BANDS`, and normalize each band to 0–1 as in the current study. Then save:

- the full-band cube (`.npy`) for the no-PCA model,
- the 3-channel PCA image (`.png`), made exactly as in the current study,
- the expert label map (`.png`).

Each image is unpacked, processed and deleted again, so disk use stays small. Re-running skips finished images.

In [ ]:
ENVI_DTYPES = {1: np.uint8, 2: np.int16, 3: np.int32, 4: np.float32, 5: np.float64,
               12: np.uint16, 13: np.uint32, 14: np.int64, 15: np.uint64}


def read_envi_header(path):
    text = open(path, 'r', errors='ignore').read()
    return {k.strip().lower(): v.strip() for k, v in
            re.findall(r'^\s*([A-Za-z][\w ]*?)\s*=\s*(\{[^}]*\}|[^\n]*)', text, flags=re.M)}


def read_envi(hdr_path):
    # Minimal ENVI reader: returns an array of shape (lines, samples, bands).
    h = read_envi_header(hdr_path)
    samples, lines, bands = int(h['samples']), int(h['lines']), int(h.get('bands', 1))
    dtype = np.dtype(ENVI_DTYPES[int(h['data type'])])
    if int(h.get('byte order', 0)) == 1:
        dtype = dtype.newbyteorder('>')
    base = hdr_path[:-4]
    data_file = next((base + e for e in ['', '.raw', '.img', '.dat', '.bin', '.bil', '.bsq', '.bip']
                      if os.path.isfile(base + e)), None)
    if data_file is None:
        raise FileNotFoundError(f"No data file next to {hdr_path}")
    count = samples * lines * bands
    data = np.fromfile(data_file, dtype=dtype, count=count, offset=int(h.get('header offset', 0)))
    if data.size != count:
        raise ValueError(f"{data_file} is smaller than its header says")
    interleave = h.get('interleave', 'bsq').lower()
    if interleave == 'bsq':
        return data.reshape(bands, lines, samples).transpose(1, 2, 0)
    if interleave == 'bil':
        return data.reshape(lines, bands, samples).transpose(0, 2, 1)
    return data.reshape(lines, samples, bands)


def reference_profile(ref, cube_shape):
    ref = ref.astype(np.float32)
    if ref.shape[1] == cube_shape[1]:
        return ref.mean(axis=0)                        # one value per column and band
    return ref.reshape(-1, ref.shape[-1]).mean(axis=0)  # one value per band


def bin_bands(cube, n):
    if cube.shape[2] <= n:
        return cube
    groups = np.array_split(np.arange(cube.shape[2]), n)
    return np.stack([cube[:, :, g].mean(axis=2) for g in groups], axis=2)


# ---- Unchanged from the current study ----
def normalize_hsi_cube(hsi):
    hsi_norm = np.zeros_like(hsi)
    for i in range(hsi.shape[2]):
        band = hsi[:, :, i]
        band_min, band_max = np.min(band), np.max(band)
        hsi_norm[:, :, i] = (band - band_min) / (band_max - band_min + 1e-8)
    return hsi_norm


def apply_pca(hsi_cube, n_components=3):
    h, w, d = hsi_cube.shape
    pca = PCA(n_components=n_components)
    return pca.fit_transform(hsi_cube.reshape(-1, d)).reshape(h, w, n_components)


def patient_of(folder):
    return re.split(r'[-_ ]', posixpath.basename(folder))[0]


for sub in ['cubes', 'pca', 'gt', 'meta', 'tmp']:
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

start, done, skipped = time.time(), 0, []
for c in CAPTURES:
    d, files, roles, campaign, name = c['folder'], c['files'], c['roles'], c['campaign'], c['name']
    if os.path.exists(os.path.join(WORK_DIR, 'meta', name + '.json')):
        continue
    tmp = os.path.join(WORK_DIR, 'tmp', name)
    try:
        with zipfile.ZipFile(c['zip']) as zf:
            wanted = [h for h in [roles['gt'], roles['cube'], roles['white'], roles['dark']] if h]
            members = [posixpath.join(d, f) for f in files if any(f == h or f.startswith(h[:-4]) for h in wanted)]
            for m in members:
                zf.extract(m, tmp)
        path = lambda f: os.path.join(tmp, *posixpath.join(d, f).split('/'))

        gt = read_envi(path(roles['gt']))[:, :, 0].astype(np.uint8)
        cube = read_envi(path(roles['cube'])).astype(np.float32)
        if gt.shape != cube.shape[:2]:
            raise ValueError(f"label map {gt.shape} and cube {cube.shape[:2]} differ in size")
        calibrated = False
        if roles['is_raw'] and roles['white'] and roles['dark']:
            white = reference_profile(read_envi(path(roles['white'])), cube.shape)
            dark = reference_profile(read_envi(path(roles['dark'])), cube.shape)
            if white.shape[-1] == cube.shape[-1]:
                cube -= dark
                cube /= np.maximum(white - dark, 1e-6)
                np.clip(cube, 0, 2, out=cube)
                calibrated = True
        bands_original = cube.shape[2]
        cube = normalize_hsi_cube(np.nan_to_num(bin_bands(cube, N_BANDS)))
        pca = apply_pca(cube.astype(np.float64))
        vis = ((pca - pca.min()) / (pca.max() - pca.min() + 1e-12) * 255).astype(np.uint8)

        np.save(os.path.join(WORK_DIR, 'cubes', name + '.npy'), cube.astype(np.float16))
        Image.fromarray(vis).save(os.path.join(WORK_DIR, 'pca', name + '.png'))
        Image.fromarray(gt).save(os.path.join(WORK_DIR, 'gt', name + '.png'))
        values, counts = np.unique(gt, return_counts=True)
        meta = {'name': name, 'campaign': campaign, 'folder': posixpath.join(os.path.basename(c['zip']), d), 'patient': patient_of(c['capture']),
                'height': int(gt.shape[0]), 'width': int(gt.shape[1]), 'bands_original': int(bands_original),
                'bands_used': int(cube.shape[2]), 'calibrated': calibrated,
                **{f'pixels_label_{int(v)}': int(c) for v, c in zip(values, counts)}}
        with open(os.path.join(WORK_DIR, 'meta', name + '.json'), 'w') as f:
            json.dump(meta, f)
        done += 1
        if done % 10 == 0:
            print(f"  {done} images prepared ({time.time() - start:.0f} s)")
    except Exception as e:
        skipped.append((name, str(e)))
        print(f"  Skipped {name}: {e}")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

print(f"Prepared {done} new images in {time.time() - start:.0f} s. Skipped {len(skipped)}.")

In [ ]:
# Dataset summary
META = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(os.path.join(WORK_DIR, 'meta', '*.json')))])
META = META.fillna(0).sort_values('name').reset_index(drop=True)
label_cols = sorted([c for c in META.columns if c.startswith('pixels_label_')], key=lambda c: int(c.split('_')[-1]))
META.to_csv(os.path.join(OUT_DIR, 'dataset_summary.csv'), index=False)

print(f"Images: {len(META)}   Patients: {META['patient'].nunique()}   Campaigns: {META['campaign'].nunique()}")
print(f"Bands: {int(META['bands_original'].iloc[0])} in the files, {int(META['bands_used'].iloc[0])} used")
print(f"Calibrated with white/dark references: {int(META['calibrated'].sum())} of {len(META)}")
print("\nLabelled pixels per label value (0 = unlabelled):")
for c in label_cols:
    v = int(c.split('_')[-1])
    print(f"  {v} {('Unlabelled' if v == 0 else CLASS_NAMES.get(v, '(unexpected value)')):15s} {int(META[c].sum()):>12,d} pixels in {int((META[c] > 0).sum())} images")
unexpected = [int(c.split('_')[-1]) for c in label_cols if int(c.split('_')[-1]) not in CLASS_NAMES and c != 'pixels_label_0']
if unexpected:
    print(f"\nWARNING: label values {unexpected} are not in CLASS_NAMES. Send this output so the label names can be checked.")
NAMES = list(META['name'])
tumour_col = f'pixels_label_{TUMOUR_CLASS}'
TUMOUR_IMAGES = list(META.loc[META[tumour_col] > 0, 'name']) if tumour_col in META else []
assert TUMOUR_IMAGES, 'No image has tumour labels. Send this output so the label values can be checked.'
print(f"\nImages with tumour labels: {len(TUMOUR_IMAGES)}")

## Part 1. Does the automatic labelling find tumour?

The current study makes its training labels automatically: SLIC superpixels, KMeans with two clusters, and then the blob nearest the image centre. Here those pseudo labels are made **with PCA** (as in the current study) and **without PCA** (all bands), and compared with the expert tumour labels. Only expert-labelled pixels count: a pseudo-label pixel on expert "tumour" is a hit, on another expert class it is a false alarm.

In [ ]:
# ---- Unchanged from the current study ----
def generate_mask(pca_image, hsi_cube, n_clusters=2):
    segments = slic(pca_image, n_segments=200, compactness=10, start_label=0)
    return mask_from_segments(segments, hsi_cube, n_clusters)


def refine_mask(mask):
    labeled = label(mask)
    regions = regionprops(labeled)
    if not regions:
        return mask
    h, w = mask.shape
    center = np.array([h / 2, w / 2])
    selected = min(regions, key=lambda r: np.linalg.norm(center - np.array(r.centroid)))
    refined = np.zeros_like(mask)
    refined[labeled == selected.label] = 1
    return refined


def apply_elliptical_roi(shape, scale=0.9):
    h, w = shape[:2]
    y, x = np.ogrid[:h, :w]
    cy, cx = h // 2, w // 2
    ry, rx = h * scale / 2, w * scale / 2
    return (((y - cy) ** 2 / ry ** 2 + (x - cx) ** 2 / rx ** 2) <= 1).astype(np.uint8)


# ---- Shared by both versions (same steps as the current study) ----
def mask_from_segments(segments, hsi_cube, n_clusters=2):
    ids = np.unique(segments)
    signatures = np.array([hsi_cube[segments == s].mean(axis=0) for s in ids])
    clusters = KMeans(n_clusters=n_clusters, random_state=42).fit_predict(signatures)
    mask = np.zeros(segments.shape, dtype=np.uint8)
    for i, s in enumerate(ids):
        mask[segments == s] = clusters[i]
    lesion = (mask == 1) if np.sum(mask == 1) < np.sum(mask == 0) else (mask == 0)
    return closing(opening(lesion.astype(np.uint8), disk(3)), disk(5))


# ---- The no-PCA version: SLIC on all bands ----
def generate_mask_no_pca(hsi_cube, n_clusters=2):
    segments = slic(hsi_cube, n_segments=200, compactness=SLIC_COMPACTNESS_NO_PCA,
                    start_label=0, channel_axis=-1, convert2lab=False)
    return mask_from_segments(segments, hsi_cube, n_clusters)


def finish(mask):
    return refine_mask(mask * apply_elliptical_roi(mask.shape))


def tumour_scores(pseudo, gt):
    labelled = np.isin(gt, list(CLASS_NAMES))
    p, t = pseudo.astype(bool)[labelled], (gt == TUMOUR_CLASS)[labelled]
    tp, fp, fn = np.sum(p & t), np.sum(p & ~t), np.sum(~p & t)
    div = lambda a, b: float(a / b) if b else 0.0
    return {'dice': div(2 * tp, 2 * tp + fp + fn), 'precision': div(tp, tp + fp), 'recall': div(tp, tp + fn)}


os.makedirs(os.path.join(WORK_DIR, 'pseudo'), exist_ok=True)
rows = []
start = time.time()
for i, name in enumerate(TUMOUR_IMAGES):
    cube = np.load(os.path.join(WORK_DIR, 'cubes', name + '.npy')).astype(np.float64)
    gt = np.array(Image.open(os.path.join(WORK_DIR, 'gt', name + '.png')))
    with_pca = finish(generate_mask(apply_pca(cube), cube))
    without_pca = finish(generate_mask_no_pca(cube))
    np.savez_compressed(os.path.join(WORK_DIR, 'pseudo', name + '.npz'), with_pca=with_pca, without_pca=without_pca)
    for version, m in [('With PCA', with_pca), ('Without PCA', without_pca)]:
        rows.append({'image': name, 'version': version, **tumour_scores(m, gt)})
    if (i + 1) % 10 == 0:
        print(f"  {i + 1} of {len(TUMOUR_IMAGES)} images ({time.time() - start:.0f} s)")

PART1 = pd.DataFrame(rows)
PART1.to_csv(os.path.join(OUT_DIR, 'part1_pseudo_labels_vs_expert.csv'), index=False)
PART1_SUMMARY = PART1.groupby('version')[['dice', 'precision', 'recall']].mean()
print(f"\nPseudo labels vs expert tumour labels, mean over {len(TUMOUR_IMAGES)} images:")
print(PART1_SUMMARY.round(3).to_string())

## Part 2. SegFormer trained on expert labels, with and without PCA

Two versions of the same SegFormer, trained with the current study's settings:

- **With PCA:** the 3-channel PCA image, prepared exactly as in the current study.
- **Without PCA:** all bands, with the first layer widened as in the melanoma comparison.

Both learn the four expert classes; unlabelled pixels are ignored. **Patient-wise cross-validation:** the images are split into folds so that no patient appears in both training and testing. Every image is predicted once, by a model that never saw that patient. Both versions use the same folds and seeds. The model after the last epoch is used, so the test images play no part in choosing it.

In [ ]:
CLASS_VALUES = sorted(CLASS_NAMES)                       # label values in the files
CLASS_LIST = [CLASS_NAMES[v] for v in CLASS_VALUES]      # names in model order
N_CLASSES = len(CLASS_VALUES)
TUMOUR_ID = CLASS_VALUES.index(TUMOUR_CLASS)
IGNORE = 255


def load_labels(name):
    gt = np.array(Image.open(os.path.join(WORK_DIR, 'gt', name + '.png')))
    lab = np.full(gt.shape, IGNORE, np.uint8)
    for i, v in enumerate(CLASS_VALUES):
        lab[gt == v] = i
    return lab


train_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=0.3), A.VerticalFlip(p=0.3), ToTensorV2()], seed=SEED)
eval_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), ToTensorV2()])
pca_norm = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


class HSIDataset(Dataset):
    def __init__(self, names, variant, transform, band_mean=None, band_std=None):
        self.names, self.variant, self.transform = names, variant, transform
        self.band_mean, self.band_std = band_mean, band_std

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        if self.variant == 'pca':
            x = np.array(Image.open(os.path.join(WORK_DIR, 'pca', name + '.png')).convert('RGB'))
            x = pca_norm(image=x)['image'].astype(np.float32)
        else:
            x = np.load(os.path.join(WORK_DIR, 'cubes', name + '.npy')).astype(np.float32)
            x = ((x - self.band_mean) / self.band_std).astype(np.float32)
        t = self.transform(image=x, mask=load_labels(name))
        return {'pixel_values': t['image'].float(), 'labels': t['mask'].long()}


def collate(batch):
    return {'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
            'labels': torch.stack([b['labels'] for b in batch])}


def band_stats(names):
    total = total_sq = 0.0
    count = 0
    for n in names:
        x = np.load(os.path.join(WORK_DIR, 'cubes', n + '.npy')).astype(np.float64)
        x = x.reshape(-1, x.shape[-1])
        total = total + x.sum(0); total_sq = total_sq + (x ** 2).sum(0); count += len(x)
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 0)) + 1e-6
    return mean.astype(np.float32), std.astype(np.float32)


def build_model(n_channels):
    model = SegformerForSemanticSegmentation.from_pretrained(
        PRETRAINED, num_labels=N_CLASSES, ignore_mismatched_sizes=True, semantic_loss_ignore_index=IGNORE,
        id2label=dict(enumerate(CLASS_LIST)), label2id={n: i for i, n in enumerate(CLASS_LIST)})
    if n_channels != 3:
        old = model.segformer.encoder.patch_embeddings[0].proj
        new = torch.nn.Conv2d(n_channels, old.out_channels, kernel_size=old.kernel_size, stride=old.stride, padding=old.padding)
        with torch.no_grad():
            new.weight.copy_(old.weight.mean(dim=1, keepdim=True).repeat(1, n_channels, 1, 1) * 3 / n_channels)
            new.bias.copy_(old.bias)
        model.segformer.encoder.patch_embeddings[0].proj = new
        model.config.num_channels = n_channels
    return model


@torch.no_grad()
def predict(model, dataset):
    model.eval().to(device)
    out = []
    for batch in DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate):
        logits = model(pixel_values=batch['pixel_values'].to(device)).logits
        pred = logits.argmax(1, keepdim=True).float()
        out.append(F.interpolate(pred, size=batch['labels'].shape[-2:], mode='nearest').squeeze(1).cpu().numpy().astype(np.uint8))
    return np.concatenate(out)


def confusion(pred, lab):
    m = lab != IGNORE
    return np.bincount(lab[m].astype(int) * N_CLASSES + pred[m].astype(int),
                       minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)


def scores_from_confusion(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    div = lambda a, b: np.divide(a, b, out=np.zeros_like(a), where=b > 0)
    iou, dice, recall = div(tp, tp + fp + fn), div(2 * tp, 2 * tp + fp + fn), div(tp, tp + fn)
    present = cm.sum(1) > 0
    return {'accuracy': float(tp.sum() / max(cm.sum(), 1)), 'mean_iou': float(iou[present].mean()) if present.any() else 0.0,
            'tumour_dice': float(dice[TUMOUR_ID]), 'tumour_recall': float(recall[TUMOUR_ID]),
            **{f'iou_{n}': float(iou[i]) for i, n in enumerate(CLASS_LIST)},
            **{f'dice_{n}': float(dice[i]) for i, n in enumerate(CLASS_LIST)}}

print(f"Classes: {CLASS_LIST}")

In [ ]:
assert META['bands_used'].nunique() == 1, 'Images have different band counts; send the dataset summary output.'
N_BANDS_USED = int(META['bands_used'].iloc[0])
VARIANTS = {'pca': 'With PCA (3 channels)', 'bands': f'Without PCA ({N_BANDS_USED} bands)'}
groups = META['patient'].values
n_folds = min(N_FOLDS, META['patient'].nunique())
FOLDS = list(GroupKFold(n_splits=n_folds).split(np.zeros(len(NAMES)), groups=groups))
EXAMPLES = sorted(TUMOUR_IMAGES, key=lambda n: -META.set_index('name').loc[n, f'pixels_label_{TUMOUR_CLASS}'])[:4]

os.makedirs(os.path.join(WORK_DIR, 'cv'), exist_ok=True)
start = time.time()
for fold, (train_idx, test_idx) in enumerate(FOLDS):
    train_names = [NAMES[i] for i in train_idx]
    test_names = [NAMES[i] for i in test_idx]
    mean, std = band_stats(train_names)
    for key, title in VARIANTS.items():
        result_file = os.path.join(WORK_DIR, 'cv', f'{key}_fold{fold}.npz')
        if os.path.exists(result_file):
            continue
        set_seed(SEED)
        train_tf.set_random_seed(SEED)
        model = build_model(3 if key == 'pca' else N_BANDS_USED)
        args = TrainingArguments(output_dir=os.path.join(WORK_DIR, 'trainer_tmp'), per_device_train_batch_size=BATCH_SIZE,
                                 num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, weight_decay=0.01, warmup_steps=50,
                                 eval_strategy='no', save_strategy='no', logging_steps=10, report_to=[],
                                 dataloader_pin_memory=False, seed=SEED, disable_tqdm=True)
        trainer = Trainer(model=model, args=args, data_collator=collate,
                          train_dataset=HSIDataset(train_names, key, train_tf, mean, std))
        trainer.train()
        test_ds = HSIDataset(test_names, key, eval_tf, mean, std)
        preds = predict(trainer.model, test_ds)
        labels = np.stack([test_ds[i]['labels'].numpy() for i in range(len(test_ds))])
        cm = confusion(preds, labels)
        keep = [i for i, n in enumerate(test_names) if n in EXAMPLES]
        np.savez_compressed(result_file, cm=cm, example_names=np.array([test_names[i] for i in keep]),
                            example_preds=preds[keep] if keep else np.zeros((0, IMG_SIZE, IMG_SIZE), np.uint8))
        s = scores_from_confusion(cm)
        print(f"Fold {fold + 1}/{n_folds} {title}: accuracy {s['accuracy']:.3f}, mean IoU {s['mean_iou']:.3f}, "
              f"tumour Dice {s['tumour_dice']:.3f}  ({(time.time() - start) / 60:.1f} min so far)")
        del trainer, model
        if device.type == 'cuda':
            torch.cuda.empty_cache()
print("Cross-validation finished.")

In [ ]:
# Collect the results
fold_rows, overall_rows, PRED_EXAMPLES = [], [], defaultdict(dict)
for key, title in VARIANTS.items():
    total = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    for fold in range(n_folds):
        r = np.load(os.path.join(WORK_DIR, 'cv', f'{key}_fold{fold}.npz'))
        total += r['cm']
        fold_rows.append({'version': title, 'fold': fold + 1, 'test_images': len(FOLDS[fold][1]), **scores_from_confusion(r['cm'])})
        for n, p in zip(r['example_names'], r['example_preds']):
            PRED_EXAMPLES[key][str(n)] = p
    overall_rows.append({'version': title, **scores_from_confusion(total)})

PART2_FOLDS = pd.DataFrame(fold_rows)
PART2 = pd.DataFrame(overall_rows).set_index('version')
PART2_FOLDS.to_csv(os.path.join(OUT_DIR, 'part2_results_per_fold.csv'), index=False)
PART2.to_csv(os.path.join(OUT_DIR, 'part2_results_overall.csv'))

print("Overall, all folds together (labelled pixels only):")
print(PART2[['accuracy', 'mean_iou', 'tumour_dice', 'tumour_recall']].round(3).to_string())
print("\nIoU per class:")
print(PART2[[f'iou_{n}' for n in CLASS_LIST]].rename(columns=lambda c: c[4:]).round(3).to_string())

## Charts

In [ ]:
colors = ['#4C72B0', '#DD8452']
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

ax = axes[0]
x = np.arange(N_CLASSES); w = 0.38
for j, title in enumerate(VARIANTS.values()):
    vals = [PART2.loc[title, f'iou_{n}'] for n in CLASS_LIST]
    bars = ax.bar(x + (j - 0.5) * w, vals, w, label=title, color=colors[j])
    ax.bar_label(bars, fmt='%.2f', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(CLASS_LIST); ax.set_ylim(0, 1); ax.set_ylabel('IoU')
ax.set_title('IoU per class (all folds)'); ax.legend(); ax.grid(axis='y', alpha=0.3)

ax = axes[1]
for j, title in enumerate(VARIANTS.values()):
    d = PART2_FOLDS[PART2_FOLDS['version'] == title]
    ax.plot(d['fold'], d['tumour_dice'], marker='o', color=colors[j], label=title)
ax.set_xticks(range(1, n_folds + 1)); ax.set_xlabel('Fold'); ax.set_ylabel('Tumour Dice'); ax.set_ylim(0, 1)
ax.set_title('Tumour Dice in each fold'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'part2_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Example images: expert labels, pseudo labels (Part 1) and SegFormer predictions (Part 2)
label_cmap = ListedColormap(['#2A9D8F', '#E76F51', '#3A86FF', '#9AA0A6', '#111111'])   # classes, then unlabelled
def label_image(lab):
    img = lab.astype(int).copy(); img[lab == IGNORE] = N_CLASSES
    return img

cols = ['PCA image', 'Expert labels', 'Pseudo label\nwith PCA', 'Pseudo label\nwithout PCA'] + [f'SegFormer\n{t}' for t in VARIANTS.values()]
fig, axes = plt.subplots(len(EXAMPLES), len(cols), figsize=(2.7 * len(cols), 2.4 * len(EXAMPLES)), squeeze=False)
for r, name in enumerate(EXAMPLES):
    pseudo = np.load(os.path.join(WORK_DIR, 'pseudo', name + '.npz'))
    lab = cv2.resize(load_labels(name), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    panels = [(np.array(Image.open(os.path.join(WORK_DIR, 'pca', name + '.png')).convert('RGB')), None),
              (label_image(lab), label_cmap), (pseudo['with_pca'], 'gray'), (pseudo['without_pca'], 'gray')]
    panels += [(label_image(PRED_EXAMPLES[k][name]), label_cmap) if name in PRED_EXAMPLES[k] else (np.zeros((2, 2)), 'gray')
               for k in VARIANTS]
    for c, (img, cmap) in enumerate(panels):
        ax = axes[r, c]
        ax.imshow(img, cmap=cmap, vmin=0 if cmap is label_cmap else None, vmax=N_CLASSES if cmap is label_cmap else None,
                  aspect='auto', interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(cols[c], fontsize=9)
    axes[r, 0].set_ylabel(name.split('__')[-1], fontsize=9)
handles = [plt.Rectangle((0, 0), 1, 1, color=label_cmap(i)) for i in range(N_CLASSES + 1)]
fig.legend(handles, CLASS_LIST + ['Unlabelled'], loc='lower center', ncol=N_CLASSES + 1, fontsize=9, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(os.path.join(OUT_DIR, 'examples.png'), dpi=150, bbox_inches='tight')
plt.show()

## Summary

The last cell prints a summary and copies every table and chart to `My Drive/HELICoiD_PCA_outputs`. **Copy the printed summary** and send it, together with the files in that folder if you can.

In [ ]:
pca_t, bands_t = VARIANTS['pca'], VARIANTS['bands']
diff = PART2_FOLDS.pivot(index='fold', columns='version', values='tumour_dice')
pca_wins = int((diff[pca_t] > diff[bands_t]).sum())
lines = [
    "HELICoiD: PCA vs no PCA with SegFormer",
    "=" * 60,
    f"Data: {len(META)} images, {META['patient'].nunique()} patients, {int(META['bands_original'].iloc[0])} bands in files, "
    f"{int(META['bands_used'].iloc[0])} used; calibrated {int(META['calibrated'].sum())}/{len(META)}",
    f"Images with tumour labels: {len(TUMOUR_IMAGES)}",
    "",
    "PART 1. Automatic pseudo labels vs expert tumour labels (mean per image)",
]
for v in PART1_SUMMARY.index:
    s = PART1_SUMMARY.loc[v]
    lines.append(f"  {v:12s} Dice {s['dice']:.3f}   precision {s['precision']:.3f}   recall {s['recall']:.3f}")
lines += ["", f"PART 2. SegFormer on expert labels, {n_folds}-fold patient-wise cross-validation, {EPOCHS} epochs"]
for v in PART2.index:
    s = PART2.loc[v]
    lines.append(f"  {v:28s} accuracy {s['accuracy']:.3f}   mean IoU {s['mean_iou']:.3f}   "
                 f"tumour Dice {s['tumour_dice']:.3f}   tumour recall {s['tumour_recall']:.3f}")
lines.append("  IoU per class:")
for n in CLASS_LIST:
    lines.append(f"    {n:15s} " + "   ".join(f"{v.split(' (')[0]}: {PART2.loc[v, f'iou_{n}']:.3f}" for v in PART2.index))
lines.append(f"  Tumour Dice per fold, with PCA:    " + ", ".join(f"{x:.3f}" for x in diff[pca_t]))
lines.append(f"  Tumour Dice per fold, without PCA: " + ", ".join(f"{x:.3f}" for x in diff[bands_t]))
lines.append(f"  PCA better in {pca_wins} of {n_folds} folds")
SUMMARY = "\n".join(lines)
print(SUMMARY)
with open(os.path.join(OUT_DIR, 'summary.txt'), 'w') as f:
    f.write(SUMMARY + "\n")

if IN_COLAB:
    with_retry(lambda: shutil.copytree(OUT_DIR, SAVE_DIR, dirs_exist_ok=True), 'saving results')
    print(f"\nResults copied to Google Drive: {SAVE_DIR}")